**Feature Engineering & Feature Store Simulation**

In [11]:
# Import required libraries

import pandas as pd
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

In [12]:
# Load CKD dataset

df = pd.read_csv("kidney_disease.csv")

print(df.head())

   id   age    bp     sg   al   su     rbc        pc         pcc          ba  \
0   0  48.0  80.0  1.020  1.0  0.0     NaN    normal  notpresent  notpresent   
1   1   7.0  50.0  1.020  4.0  0.0     NaN    normal  notpresent  notpresent   
2   2  62.0  80.0  1.010  2.0  3.0  normal    normal  notpresent  notpresent   
3   3  48.0  70.0  1.005  4.0  0.0  normal  abnormal     present  notpresent   
4   4  51.0  80.0  1.010  2.0  0.0  normal    normal  notpresent  notpresent   

   ...  pcv    wc   rc  htn   dm  cad appet   pe  ane classification  
0  ...   44  7800  5.2  yes  yes   no  good   no   no            ckd  
1  ...   38  6000  NaN   no   no   no  good   no   no            ckd  
2  ...   31  7500  NaN   no  yes   no  poor   no  yes            ckd  
3  ...   32  6700  3.9  yes   no   no  poor  yes  yes            ckd  
4  ...   35  7300  4.6   no   no   no  good   no   no            ckd  

[5 rows x 26 columns]


**Basic Cleaning**

In [14]:
# Remove id column if present

if 'id' in df.columns:
    df.drop('id', axis=1, inplace=True)

# Replace '?' with missing values

df.replace('?', pd.NA, inplace=True)

# Convert numeric columns

numeric_columns = ['age','bp','sg','al','su','bgr','bu','sc','sod','pot','hemo','pcv','wc','rc']

for col in numeric_columns:

    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

**Create Derived Features**

1 Kidney Risk Score

Higher values indicate higher kidney risk.

In [15]:
df["kidney_risk_score"] = (

    df["bu"] +

    df["sc"] +

    df["al"]

)

2 High Blood Pressure Flag

In [16]:
df["high_bp"] = (

    df["bp"] > 90

).astype(int)

3 Anaemia Flag

In [17]:
df["anemia"] = (

    df["hemo"] < 12

).astype(int)

4 Blood Glucose Category

In [18]:
df["high_glucose"] = (

    df["bgr"] > 140

).astype(int)

**Create Aggregated Feature**

In [20]:
#Create a simple overall health indicator.
df["health_score"] = (

    df["hemo"] +

    df["pcv"]

)

**Display Newly Created Features**

In [21]:
print(df[[
    "kidney_risk_score",
    "high_bp",
    "anemia",
    "high_glucose",
    "health_score"
]].head())

   kidney_risk_score  high_bp  anemia  high_glucose  health_score
0               38.2        0       0             0          59.4
1               22.8        0       1             0          49.3
2               56.8        0       1             1          40.6
3               63.8        0       1             0          43.2
4               29.4        0       1             0          46.6


**Simulate Feature Store**

In [22]:
# Create folder

os.makedirs("feature_store", exist_ok=True)

# Save engineered features

engineered_features = df[[
    "kidney_risk_score",
    "high_bp",
    "anemia",
    "high_glucose",
    "health_score"
]]

engineered_features.to_csv(

    "feature_store/ckd_engineered_features.csv",

    index=False

)

print("Feature Store Created")

Feature Store Created


**Reuse Feature Store**

In [23]:
stored_features = pd.read_csv(

    "feature_store/ckd_engineered_features.csv"

)

print(stored_features.head())

   kidney_risk_score  high_bp  anemia  high_glucose  health_score
0               38.2        0       0             0          59.4
1               22.8        0       1             0          49.3
2               56.8        0       1             1          40.6
3               63.8        0       1             0          43.2
4               29.4        0       1             0          46.6


**Compare Models**

Model 1 – Original Dataset

In [24]:
original_df = df.copy()

X_original = original_df.drop("classification", axis=1)

y = original_df["classification"]

Model 2 – Engineered Dataset

In [25]:
engineered_df = pd.concat(

    [df, stored_features],

    axis=1

)

X_engineered = engineered_df.drop(

    "classification",

    axis=1

)